# Comparación de modelos de sentimiento financiero

Cuaderno autónomo reconstruido a partir de las salidas y código del PDF original. Ejecutar en Colab con GPU. Los resultados nuevos se calculan al ejecutar; las cifras originales están en el repositorio y no se presentan como salidas nuevas. No realizo fine-tuning. Algunos modelos se ajustaron con Financial PhraseBank, por lo que la evaluación es exploratoria.

## 1 Configuración de GPU
Activo GPU desde Entorno de ejecución → Cambiar tipo de entorno de ejecución.

In [ ]:
import torch
device = 0 if torch.cuda.is_available() else -1
print("GPU: " + torch.cuda.get_device_name(0) if device == 0 else "Ejecutando en CPU; los tiempos no serán comparables con Tesla T4")

## 2 Instalación de dependencias
Instalo antes de importar Transformers. Si Colab solicita reiniciar, lo hago y continúo.

In [ ]:
%pip install -q "transformers==4.57.1" "huggingface_hub==0.36.0" kagglehub scikit-learn pandas numpy

## 3 Autenticación en Hugging Face
Introduzco un token de lectura en el campo oculto. Los modelos son públicos; puedo dejar el campo vacío para continuar sin autenticar.

In [ ]:
from getpass import getpass
from huggingface_hub import login
token = getpass("Token de Hugging Face (vacío para omitir): ")
if token:
    login(token=token, add_to_git_credential=False)
del token

## 4 Carga y preparación del dataset
Descargo Kaggle versión 5 y selecciono 1,000 textos con semilla 110. No requiere archivos externos de src.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import kagglehub

def descargar():
    # Fijo la versión 5, correspondiente a la descarga documentada.
    identificador = "ankurzing/sentiment-analysis-for-financial-news/versions/5"
    ruta = Path(kagglehub.dataset_download(identificador))
    destino = Path("data/raw")
    destino.mkdir(parents=True, exist_ok=True)
    archivo = destino / "all-data.csv"
    shutil.copy2(ruta / "all-data.csv", archivo)
    if (ruta / "FinancialPhraseBank").exists():
        shutil.copytree(ruta / "FinancialPhraseBank", destino / "FinancialPhraseBank", dirs_exist_ok=True)
    metadatos = {
        "fuente": "Kaggle", "identificador": identificador,
        "archivo": "all-data.csv",
        "sha256": hashlib.sha256(archivo.read_bytes()).hexdigest()
    }
    (destino / "origen_dataset.json").write_text(
        json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("Dataset descargado:", archivo)
    return archivo


archivo = descargar()

In [ ]:
from pathlib import Path
import pandas as pd

def preparar(archivo="data/raw/all-data.csv"):
    # Replico la limpieza y el muestreo del notebook documentado.
    df = pd.read_csv(archivo, encoding="latin-1", header=None, names=["sentimiento", "texto"])
    df = df.dropna().drop_duplicates(subset="texto").copy()
    df["sentimiento"] = df["sentimiento"].str.strip().str.lower()
    df["texto"] = df["texto"].str.strip()
    datos = df.sample(n=min(1000, len(df)), random_state=110).reset_index(drop=True)
    destino = Path("data/processed")
    destino.mkdir(parents=True, exist_ok=True)
    df.to_csv(destino / "noticias_limpias.csv", index=False)
    datos.to_csv(destino / "muestra_evaluacion.csv", index=False)
    print("Noticias disponibles:", len(df))
    print("Noticias que evaluaré:", len(datos))
    print(datos["sentimiento"].value_counts())
    return datos


datos = preparar(archivo)
display(datos.head())

In [ ]:
from pathlib import Path
import pandas as pd

def resumir():
    datos = pd.read_csv("data/processed/muestra_evaluacion.csv")
    resumen = datos["sentimiento"].value_counts().rename_axis("sentimiento").reset_index(name="cantidad")
    resumen["porcentaje"] = (resumen["cantidad"] / len(datos) * 100).round(2)
    Path("results").mkdir(exist_ok=True)
    resumen.to_csv("results/distribucion_dataset.csv", index=False)
    print(resumen.to_string(index=False))
    return resumen


display(resumir())

## 5 Pipeline de inferencia
## 6 Métricas
## 7 Latencia
Defino la función común para cargar cada modelo, unificar etiquetas, medir 30 solicitudes individuales y clasificar la muestra por lotes. Cada modelo se carga y libera por separado. Las métricas macro dan igual peso a las tres clases.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import gc
import json
import time
import importlib.metadata as metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODELOS = {
    "FinBERT": "ProsusAI/finbert",
    "FinBERT Tone": "yiyanghkust/finbert-tone",
    "DistilRoBERTa": "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis"
}
ETIQUETAS = {
    "FinBERT Tone": {"label_0": "neutral", "label_1": "positive", "label_2": "negative"},
    "DistilRoBERTa": {"label_0": "negative", "label_1": "neutral", "label_2": "positive"}
}

def evaluar(datos):
    # Utilizo la misma muestra y configuración para los tres modelos.
    device = 0 if torch.cuda.is_available() else -1
    textos = datos["texto"].tolist()
    reales = datos["sentimiento"].tolist()
    resultados, predicciones = [], []
    revisiones = {}
    for nombre, identificador in MODELOS.items():
        print("Evaluando", nombre)
        clasificador = pipeline("text-classification", model=identificador, tokenizer=identificador, device=device)
        revisiones[nombre] = getattr(clasificador.model.config, "_commit_hash", None)
        # Caliento el modelo antes de medir las solicitudes individuales.
        clasificador(textos[0], truncation=True, max_length=128)
        tiempos = []
        for texto in textos[:30]:
            if device == 0: torch.cuda.synchronize()
            inicio = time.perf_counter()
            clasificador(texto, truncation=True, max_length=128)
            if device == 0: torch.cuda.synchronize()
            tiempos.append((time.perf_counter() - inicio) * 1000)
        if device == 0: torch.cuda.synchronize()
        inicio = time.perf_counter()
        salidas = clasificador(textos, batch_size=16, truncation=True, max_length=128)
        if device == 0: torch.cuda.synchronize()
        tiempo_total = time.perf_counter() - inicio
        mapa = ETIQUETAS.get(nombre, {})
        pred = [mapa.get(s["label"].lower(), s["label"].lower()) for s in salidas]
        precision, recall, f1, _ = precision_recall_fscore_support(
            reales, pred, labels=["negative", "neutral", "positive"], average="macro", zero_division=0
        )
        resultados.append({
            "Modelo": nombre, "Accuracy": accuracy_score(reales, pred),
            "Precision macro": precision, "Recall macro": recall, "F1 macro": f1,
            "Latencia media (ms)": float(np.mean(tiempos)),
            "Tiempo total por lotes (s)": tiempo_total
        })
        for texto, real, etiqueta, salida in zip(textos, reales, pred, salidas):
            predicciones.append({"Modelo": nombre, "texto": texto, "sentimiento": real,
                                 "prediccion": etiqueta, "confianza": salida["score"]})
        del clasificador
        gc.collect()
        if device == 0: torch.cuda.empty_cache()
    tabla = pd.DataFrame(resultados).sort_values("F1 macro", ascending=False).reset_index(drop=True)
    Path("results").mkdir(exist_ok=True)
    # Los resultados nuevos tienen otro nombre para conservar la evidencia original.
    tabla.to_csv("results/comparacion_modelos_nueva.csv", index=False)
    pd.DataFrame(predicciones).to_csv("results/predicciones_nuevas.csv", index=False)
    entorno = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "hardware": torch.cuda.get_device_name(0) if device == 0 else "CPU",
        "registros": len(datos), "semilla": 110, "batch_size": 16, "max_length": 128,
        "textos_latencia": min(30, len(datos)), "revisiones_modelos": revisiones,
        "versiones": {p: metadata.version(p) for p in ["torch", "transformers", "huggingface_hub", "kagglehub", "pandas", "numpy", "scikit-learn"]}
    }
    Path("results/entorno_nuevo.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False), encoding="utf-8")
    return tabla

In [ ]:
tabla = evaluar(datos)

## 8 Tabla comparativa
Consulto los resultados calculados en esta ejecución.

In [ ]:
display(tabla.round(4))
predicciones = pd.read_csv("results/predicciones_nuevas.csv")
display(predicciones.groupby("Modelo", sort=False).head(5))

## 9 Conclusiones parciales
Comparo calidad y latencia; no interpreto la confianza como probabilidad calibrada.

In [ ]:
mejor = tabla.iloc[0]
rapido = tabla.loc[tabla["Latencia media (ms)"].idxmin()]
lotes = tabla.loc[tabla["Tiempo total por lotes (s)"].idxmin()]
print(f"Mayor F1 macro: {mejor['Modelo']}, {mejor['F1 macro']:.4f}")
print(f"Menor latencia individual: {rapido['Modelo']}")
print(f"Menor tiempo por lotes: {lotes['Modelo']}")
print("Recomendación por calidad: provisional y pendiente de evaluar noticias independientes.")

## Descargar los resultados nuevos
Guardo CSV, predicciones y condiciones de ejecución para incorporarlos al repositorio.

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile("resultados_nuevos.zip", "w", zipfile.ZIP_DEFLATED) as paquete:
    for nombre in ["comparacion_modelos_nueva.csv", "predicciones_nuevas.csv", "entorno_nuevo.json", "distribucion_dataset.csv"]:
        archivo = Path("results") / nombre
        paquete.write(archivo, arcname=nombre)
files.download("resultados_nuevos.zip")